# Tool Thinness Distribution

**Purpose**: compute the tool-thinness distribution across labelled masks (mean instrument width in pixels, from the distance-transform inscribed radius),
and report percentiles for this notebook's resize-first thin/wide mark.

**Input**: Pascal VOC segmentation masks (`SegmentationClass/`) plus `labelmap.txt`.
Uses local paths when present; otherwise pulls from GCS (`gs://laparoscopic-seg-REPLACE_K_NUMBER/trainingFrames/`).

**Output**: percentile table + histogram of mean tool width, then correlation with area and slenderness as a rejected alternative.


## 1. Install dependencies
Run this cell to install the packages required for this notebook.

In [ ]:
%pip install -q matplotlib tqdm pandas opencv-python-headless scipy

import cv2, matplotlib, pandas as pd, numpy as _np, scipy
print(f"cv2 {cv2.__version__} | matplotlib {matplotlib.__version__} "
      f"| pandas {pd.__version__} | numpy {_np.__version__} | scipy {scipy.__version__}")

## 2. Setup and configuration
Resolves the mask directory and labelmap path (local first, GCS fallback), then sets the
foreground label and the model-input size used for width measurement. Override with
environment variables if the files move.

In [ ]:
import os
import glob
import shutil
import subprocess
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt
from scipy.stats import spearmanr

# ── Paths ────────
# Override with environment variables if the files move.
GCS_BUCKET = os.getenv("LAP_GCS_BUCKET", "laparoscopic-seg-REPLACE_K_NUMBER")
GCS_PREFIX = f"gs://{GCS_BUCKET}/trainingFrames"
CACHE_DIR = Path(os.getenv("LAP_CACHE_DIR", Path.home() / ".cache" / "laparoscopic_seg"))

DEFAULT_DATA_ROOT = Path.home() / "Desktop" / "Individual Project" / "combined"
MASKS_DIR = Path(os.getenv("LAP_MASKS_DIR", DEFAULT_DATA_ROOT / "SegmentationClass"))
FOREGROUND_LABEL = os.getenv("LAP_FOREGROUND_LABEL", "instrument")
TARGET = 512  # model input; measure width after INTER_NEAREST resize, not at native


def _count_pngs(root: Path) -> int:
    return len(glob.glob(str(root / "**" / "*.png"), recursive=True))


def _gsutil_available() -> bool:
    return shutil.which("gsutil") is not None


def _gsutil_cp(src: str, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gsutil", "cp", src, str(dst)], check=True)


def _gsutil_rsync(src: str, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gsutil", "-m", "rsync", "-r", src, str(dst)], check=True)


def resolve_labelmap() -> Path:
    """Local candidates first; download from GCS into the cache if none exist."""
    if os.getenv("LAP_LABELMAP"):
        candidate = Path(os.getenv("LAP_LABELMAP"))
        if candidate.is_file():
            return candidate
        raise FileNotFoundError(f"LAP_LABELMAP set but not found: {candidate}")

    for candidate in [
        MASKS_DIR.parent / "labelmap.txt",
        Path.cwd() / "labelmap.txt",
        Path.home() / "VSCodeProjects" / "dissertation" / "archive" / "examples" / "labelmap.txt",
        DEFAULT_DATA_ROOT / "labelmap.txt",
    ]:
        if candidate.is_file():
            return candidate

    if not _gsutil_available():
        raise FileNotFoundError(
            "labelmap.txt not found locally and gsutil is unavailable.\n"
            "Set LAP_LABELMAP, or install the Google Cloud SDK."
        )
    cached = CACHE_DIR / "labelmap.txt"
    print(f"Downloading labelmap from {GCS_PREFIX}/labelmap.txt ...")
    _gsutil_cp(f"{GCS_PREFIX}/labelmap.txt", cached)
    return cached


def resolve_masks_dir() -> Path:
    """Use local masks if present; otherwise rsync SegmentationClass from GCS."""
    if MASKS_DIR.is_dir() and _count_pngs(MASKS_DIR) > 0:
        return MASKS_DIR

    if os.getenv("LAP_MASKS_DIR") and not MASKS_DIR.is_dir():
        raise FileNotFoundError(f"LAP_MASKS_DIR set but not found: {MASKS_DIR}")

    if not _gsutil_available():
        raise FileNotFoundError(
            f"Masks dir not found (or empty): {MASKS_DIR}\n"
            "Set LAP_MASKS_DIR, or install gsutil to pull from GCS."
        )
    cached = CACHE_DIR / "SegmentationClass"
    if _count_pngs(cached) == 0:
        print(f"Downloading masks from {GCS_PREFIX}/SegmentationClass ...")
        _gsutil_rsync(f"{GCS_PREFIX}/SegmentationClass", cached)
    else:
        print(f"Using cached masks ({_count_pngs(cached)} pngs): {cached}")
    return cached


LABELMAP_FILE = resolve_labelmap()
MASKS_DIR = resolve_masks_dir()

n_masks = _count_pngs(MASKS_DIR)
print("Masks dir:", MASKS_DIR)
print("Labelmap:", LABELMAP_FILE)
print(f"Foreground label: {FOREGROUND_LABEL}")
print(f"Width measured at: {TARGET}x{TARGET}")
print(f"Mask PNGs found: {n_masks}")
assert n_masks > 0, f"No .png masks under {MASKS_DIR}"
assert LABELMAP_FILE.is_file(), f"Labelmap not found: {LABELMAP_FILE}"

## 3. Parse labelmap
Reads `labelmap.txt` to map class names to RGB colours. The colour corresponding to
`FOREGROUND_LABEL` is used for the binary mask conversion.

In [ ]:
def parse_labelmap(path: Path) -> dict:
    """Return {label_name: (r, g, b)} from a Pascal VOC segmentation format (`name:r,g,b::`)."""
    label_colours = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            name, rgb_part, *_ = line.split(":")
            r, g, b = [int(x) for x in rgb_part.split(",")]
            label_colours[name] = (r, g, b)
    return label_colours

label_colours = parse_labelmap(LABELMAP_FILE)
print("=" * 60)
for name, (r, g, b) in label_colours.items():
    print(f"\033[48;2;{r};{g};{b}m    \033[0m {name:12s} : ({r}, {g}, {b})")

assert FOREGROUND_LABEL in label_colours, (
    f"'{FOREGROUND_LABEL}' not in labelmap. Available: {list(label_colours)}"
)

FG_COLOUR = np.array(label_colours[FOREGROUND_LABEL], dtype=np.uint8)
print(f"foreground     : '{FOREGROUND_LABEL}' RGB{tuple(FG_COLOUR)}")
print("=" * 60)

## 4. Tool thinness distribution (mean width at model input)
Scan every mask, resize to `TARGET` with `INTER_NEAREST`, measure mean local width via the
distance-transform inscribed radius (doubled to a full width), then print a percentile table and the
24 px mark (p25 rounded) and plot the histogram.


In [ ]:
mask_files = glob.glob(str(MASKS_DIR / "**" / "*.png"), recursive=True)

widths = []
sizes = []
area_px = []

print(f"Calculating widths for {len(mask_files)} masks...")
for mask_file in tqdm(mask_files):
    mask_rgb = cv2.cvtColor(cv2.imread(mask_file), cv2.COLOR_BGR2RGB)
    fg = np.all(mask_rgb == FG_COLOUR, axis=-1)
    if not fg.any():
        continue
    m = cv2.resize(fg.astype(np.uint8), (TARGET, TARGET), interpolation=cv2.INTER_NEAREST) > 0
    if not m.any():
        continue
    widths.append(2.0 * distance_transform_edt(m)[m].mean())
    sizes.append(fg.mean())
    area_px.append(int(m.sum()))

widths_series = pd.Series(widths)
sizes_series = pd.Series(sizes) * 100
NOTEBOOK_CUT = 24.0  # p25 (24.1) rounded


def dist_table(series, name, decimals=2):
    s = series.describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90])
    df = pd.DataFrame([{
        "n": int(s["count"]),
        "mean": s["mean"],
        "std": s["std"],
        "min": s["min"],
        "p5": s["5%"],
        "p10": s["10%"],
        "p25": s["25%"],
        "p50": s["50%"],
        "p75": s["75%"],
        "p90": s["90%"],
        "max": s["max"],
    }])
    print(f"\n--- {name} ---")
    fmt = lambda v, d=decimals: f"{v:.{d}f}"
    print(df.to_string(index=False, formatters={
        "n": "{:.0f}".format,
        **{c: fmt for c in df.columns if c != "n"},
    }))
    return s


stats = dist_table(widths_series, "Tool Thinness Distribution (Mean Width at 512x512, px)")
print(f"plot line: {NOTEBOOK_CUT:.0f} px (p25 rounded, p25 = {stats['25%']:.1f})")

plt.figure(figsize=(10, 5))
plt.hist(widths_series, bins=50, color='red', edgecolor='white')
plt.title("Distribution of Mean Tool Widths")
plt.xlabel("Mean Local Width (px at 512x512)")
plt.ylabel("Frequency (Frames)")
plt.axvline(NOTEBOOK_CUT, color='#333333', linestyle='dashed', linewidth=2, label='Notebook mark (24 px)')
plt.legend()
plt.show()


## 5. Rejected alternatives
This notebook's 24 px mark (resize first, then measure width), plus two checks that thinness is not just size under another name. The runs cut at 22.81 px on a different width implementation.

1. **Overlap with size** - whether mean width and tool area move together. If they do, "thin" would mostly be the same frames as "tiny".
2. **Slenderness** - width divided by the square root of area. This measures shape, not the pixel width the network sees, so it was rejected.


In [ ]:
n = len(widths_series)
mx = float(widths_series.max())


def band_table(edges, name):
    counts, e = np.histogram(widths_series, bins=edges)
    rows = []
    for i, c in enumerate(counts):
        closed = "]" if i == len(counts) - 1 else ")"
        rows.append({
            "band": f"[{e[i]:.1f}, {e[i + 1]:.1f}{closed}",
            "frames": int(c),
            "pct": 100.0 * c / n,
        })
    df = pd.DataFrame(rows)
    print(f"\n--- {name} ---")
    print(df.to_string(index=False, formatters={"pct": "{:.1f}%".format}))
    i_hi, i_lo = int(df["frames"].idxmax()), int(df["frames"].idxmin())
    print(f"densest: {df.loc[i_hi, 'pct']:.1f}%   sparsest: {int(df.loc[i_lo, 'frames'])} frames")
    return df


_ = band_table(np.array([0.0, NOTEBOOK_CUT, mx]), "Notebook mark (24 px thin / wide)")

rho, pval = spearmanr(widths_series, sizes_series)
print("\n--- Correlation of mean width vs area % ---")
print(f"Spearman correlation: {rho:.3f}  p={'<1e-12' if pval < 1e-12 else f'{pval:.2g}'}")
if abs(rho) >= 0.8:
    print("correlation >= 0.8: thin is largely a relabelling of tiny. "
          "Report as a partially overlapping axis, or use slenderness.")

slenderness = widths_series / np.sqrt(pd.Series(area_px))
_ = dist_table(slenderness, "Slenderness (width / sqrt(area_px) at 512x512)", decimals=3)
rho_s, p_s = spearmanr(slenderness, sizes_series)
print(f"Spearman correlation vs area %: {rho_s:.3f}  p={'<1e-12' if p_s < 1e-12 else f'{p_s:.2g}'}")
